<a href="https://colab.research.google.com/github/RCDS13/PI-JP-4SEM/blob/main/etl_transito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)
!ls "/content/drive/MyDrive/PI_JP"

!pip install rpy2
%load_ext rpy2.ipython

In [ ]:
%%R

library(dplyr)
library(stringi)
library(tidyverse)


In [ ]:
%%R

path <- "/content/drive/MyDrive/PI_JP/"

# PADRONIZACAO E LEITURA
df1 <- read_delim(paste0(path, "datatran2021.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df2 <- read_delim(paste0(path, "datatran2022.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df3 <- read_delim(paste0(path, "datatran2023.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df4 <- read_delim(paste0(path, "datatran2024.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))
df5 <- read_delim(paste0(path, "datatran2025.csv"), delim = ";", locale = locale(encoding = "latin1")) %>% mutate(across(any_of(c("latitude", "longitude")), as.character))

# JUNCAO DE TABELAS
df_final <- bind_rows(df1, df2, df3, df4, df5)

# SALVAR TABELA FINAL
write_excel_csv(df_final, paste0(path, "dataset_final.csv"), delim = ";")


In [8]:
%%R
# VERIFICA LINHAS DUPLICADAS

total_original <- nrow(df_final)

total_unicas <- nrow(distinct(df_final))

duplicados <- total_original - total_unicas

print(paste("Registros duplicados encontrados:", duplicados))

[1] "Registros duplicados encontrados: 0"


In [9]:
%%R

# CARREGA TABELA FINAL

dados <- read.csv2("/content/drive/MyDrive/PI_JP/dataset_final.csv", fileEncoding="UTF-8")

# REMOCAO DE ACENTOS
dados_sp <- dados %>%
  filter(uf == "SP") %>%
  mutate(across(where(is.character), ~ stri_trans_general(., "Latin-ASCII")))

# SALVAMENTO USANDO FILE ENCONDING LATIN1 PRA COMTABILIDADE
write.csv2(dados_sp,
           "/content/drive/MyDrive/PI_JP/dataset_final_SP.csv",
           row.names = FALSE,
           fileEncoding = "latin1")

In [ ]:
%%R

# ESCOPO DO DICINARIO DE VARIAVEIS
dicionario <- data.frame(
  Variavel = names(dados),
  Tipo_R = sapply(dados, class),
  Descricao = "",
  Valores = "",
  stringsAsFactors = FALSE
)

# VISUALIZACAO
print(dicionario)

# SALVA NO DRIVE
write.csv2(dicionario,
           "/content/drive/MyDrive/PI_JP/dicionario_variaveis.csv",
           row.names = FALSE,
           fileEncoding = "UTF-8")

In [10]:
%%R

names(dados_sp)

 [1] "id"                     "data_inversa"           "dia_semana"            
 [4] "horario"                "uf"                     "br"                    
 [7] "km"                     "municipio"              "causa_acidente"        
[10] "tipo_acidente"          "classificacao_acidente" "fase_dia"              
[13] "sentido_via"            "condicao_metereologica" "tipo_pista"            
[16] "tracado_via"            "uso_solo"               "pessoas"               
[19] "mortos"                 "feridos_leves"          "feridos_graves"        
[22] "ilesos"                 "ignorados"              "feridos"               
[25] "veiculos"               "latitude"               "longitude"             
[28] "regional"               "delegacia"              "uop"                   


In [35]:
%%R

media_mortos <- mean(dados_sp$mortos, na.rm = TRUE)
media_ilesos <- mean(dados_sp$ilesos, na.rm = TRUE)
media_veiculos <- mean(dados_sp$veiculos, na.rm = TRUE)
media_feridos_leves <- mean(dados_sp$feridos_leves, na.rm = TRUE)
media_feridos_graves <- mean(dados_sp$feridos_graves, na.rm = TRUE)
media_feridos <- mean(dados_sp$feridos, na.rm = TRUE)
media_pessoas <- mean(dados_sp$pessoas, na.rm = TRUE)

cat("média de óbitos:", media_mortos, "\n")
cat("média de ilesos:",media_ilesos, "\n")
cat("média de veículos:",media_veiculos, "\n")
cat("média de feridos leves:",media_feridos_leves, "\n")
cat("média de feridos graves:",media_feridos_graves, "\n")
cat("média de feridos:",media_feridos, "\n")
cat("média de pessoas:",media_pessoas)


média de óbitos: 0.04874652 
média de ilesos: 1.067977 
média de veículos: 1.948414 
média de feridos leves: 0.8991322 
média de feridos graves: 0.1668095 
média de feridos: 1.065942 
média de pessoas: 2.473645